> Historical development experiment. Read [notebook20](20_practical_options_report.ipynb) for the final recommendation.

## Recommendation: separate animal detection from context detection

Our existing **DroneMegaDetector remains the strongest seal proposer in this pilot**. At confidence 0.3 with full-frame plus tiles, it matched **10/14** labeled seals; Grounding DINO Tiny matched **2/14**, YOLO-World v2 **0/14**, and YOLOE-26s **6/14**. Unmatched animal proposals were 23, 37, 18 and 86 respectively. This is a small previously inspected regression set, not a held-out benchmark.

**YOLO-World is more promising for the context branch:** on eight labeled COCO controls, full-frame inference matched **23/30 people and 5/6 vehicles**, versus DMD's **11/30 and 0/6**. These are general-image controls; aerial seal-plus-human retention still needs validation.

I built an **offline role-separated proposal prototype**: DMD supplies animal candidates; YOLO-World supplies person/vehicle candidates. Its composed outputs preserve 10/14 seal matches and the 23/30 person, 5/6 vehicle matches on these same inputs. It always requires animal confirmation before context display. Person overlap is a review hint, not a reason to delete an animal proposal. Gemini was not called in this experiment.

YOLO-World's whole-frame median was **0.064 seconds** on eight sampled 4K frames from two clips. DMD's full-plus-tile median was **2.080 seconds** on those frames; the animal branch remains the speed bottleneck. Separate-run timings do not establish combined live throughput.

**Next layer design to validate:** preserve the full overview, select diverse native-resolution animal crops, and verify short crop sequences with Gemini. Keep uncertain candidates available for review. A model swap alone did not solve small-seal detection, false animal proposals, or the crop-budget problem. No production integration, fine-tuning or deployment.

# Alternative detector comparison

Exploratory GPU comparison of our current DroneMegaDetector against open-vocabulary alternatives. The same locked sources and 1280-pixel tile windows were used; native model preprocessing differs. **No Gemini calls, fine-tuning or deployment.**

## Labeled seals: full image plus tiles

| Model | Matched seals / 14 | Unmatched animal proposals | Matched in top 3 | Median seconds / large NOAA image |
|---|---:|---:|---:|---:|
| dmd | 10/14 | 23 | 10/14 | 5.73 |
| grounding_dino | 2/14 | 37 | 2/14 | 8.86 |
| yolo_world | 0/14 | 18 | 0/14 | 1.22 |
| yoloe26 | 6/14 | 86 | 2/14 | 1.64 |

Eight previously inspected NOAA sources, fourteen reference boxes. IoU0.5 matching; fixed confidence0.3. This is a small regression comparison, not held-out accuracy. Unmatched proposals are defined against available annotations. Confidence scales differ between models.

## Person/vehicle retention on labeled COCO controls

| Model | Class | Matched / references | Unmatched proposals |
|---|---|---:|---:|
| dmd | person | 11/30 | 1 |
| dmd | vehicle | 0/6 | 0 |
| grounding_dino | person | 25/30 | 9 |
| grounding_dino | vehicle | 6/6 | 1 |
| yolo_world | person | 23/30 | 2 |
| yolo_world | vehicle | 5/6 | 0 |
| yoloe26 | person | 19/30 | 3 |
| yoloe26 | vehicle | 4/6 | 1 |

These controls test proposal retention only, before any animal-presence display gate. They are general COCO images, not aerial rescue footage.

## Practical video runtime (mixed 1080p and 4K inputs)

| Model | Whole-frame median / sampled video frame | Full+tiles median |
|---|---:|---:|
| dmd | 0.235 s | 1.375 s |
| grounding_dino | 0.521 s | 2.285 s |
| yolo_world | 0.044 s | 0.266 s |
| yoloe26 | 0.047 s | 0.337 s |

Warm synchronized GPU timing excludes disk decode, model downloads and loading. It includes native inference, image crops and merge. It does not establish live-stream throughput or end-to-end Gemini latency.

## What this does and does not answer

The model swap must improve animal proposal coverage without flooding Gemini or losing people/vehicles. Read matched-in-top-three alongside total matches: a high-recall pool is not enough if the evidence budget excludes the true animal. Video examples show proposed-animal crops; the animals and contextual boxes in those clips are not exhaustively annotated.

Threshold sweeps, per-class metrics, exploratory DMD+challenger unions, source hashes, weights/revisions, model versions and full predictions are archived. Hybrid confidence scores are not calibrated. No single winner should be promoted from fourteen seal boxes.

## Sources for the model choices

- [DroneMegaDetector](https://github.com/ConservationDronesAI/DroneMegaDetector)
- [Grounding DINO](https://huggingface.co/docs/transformers/model_doc/grounding-dino)
- [YOLO-World](https://docs.ultralytics.com/models/yolo-world/)
- [YOLOE](https://docs.ultralytics.com/models/yoloe/)

## 4K-only runtime subset

| Model | Full frame | Full+tiles |
|---|---:|---:|
| dmd | 0.248 s | 2.080 s |
| grounding_dino | 0.548 s | 3.356 s |
| yolo_world | 0.064 s | 0.422 s |
| yoloe26 | 0.067 s | 0.551 s |

Eight 4K samples from two clips. The earlier mixed-resolution table includes eight 1080p samples as well.


![Research evidence](../assets/seal-model-comparison.jpg)

Attribution: [CREDITS.json](../assets/CREDITS.json).

In [ ]:
from pathlib import Path
import json
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'research/aerial_detector/results').is_dir())
research=root/'research/aerial_detector'
print(json.dumps(json.loads((research/'results/model-comparison-v1/report.json').read_text()),indent=2))